# SmolVLA Local Sanity Check

Run this AFTER `install_vla_env.sh` has completed successfully.

Before opening this notebook, activate the environment in your terminal:
```bash
conda activate vla-interp
cd ~/vla-interp-project
jupyter lab
```
Then open this notebook from the JupyterLab file browser, or in VS Code with the `vla-interp` interpreter selected.

## 1. Confirm environment

In [1]:
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Running on CPU only (expected, since this is a local AMD-GPU machine).')

torch version: 2.11.0+cu130
CUDA available: False
Running on CPU only (expected, since this is a local AMD-GPU machine).


## 2. Load SmolVLA checkpoint

Correct import path (no `.common` -- LeRobot dropped that in a recent refactor).

In [2]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained('lerobot/smolvla_libero')
print(policy)
# SAVE THIS OUTPUT -- you'll need layer/dim counts later for the capacity comparison against Pi0.5

/home/rithvik/miniconda3/envs/vla-interp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████| 489/489 [00:00<00:00, 497.96it/s]


Reducing the number of VLM layers to 16 ...
SmolVLAPolicy(
  (model): VLAFlowMatching(
    (vlm_with_expert): SmolVLMWithExpertModel(
      (vlm): SmolVLMForConditionalGeneration(
        (model): SmolVLMModel(
          (vision_model): SmolVLMVisionTransformer(
            (embeddings): SmolVLMVisionEmbeddings(
              (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
              (position_embedding): Embedding(1024, 768)
            )
            (encoder): SmolVLMEncoder(
              (layers): ModuleList(
                (0-11): 12 x SmolVLMEncoderLayer(
                  (self_attn): SmolVLMVisionAttention(
                    (k_proj): Linear(in_features=768, out_features=768, bias=True)
                    (v_proj): Linear(in_features=768, out_features=768, bias=True)
                    (q_proj): Linear(in_features=768, out_features=768, bias=True)
                    (out_proj): Linear(in_features=768, out_features=768, bias=True)

## 3. Jacobian/JVP sanity check (torch.func)

Confirms the core tool your Jacobian Lens computation depends on works in this environment.

In [3]:
from torch.func import jacrev

def f(x):
    return x.sum(dim=-1)

x = torch.randn(4, 8, requires_grad=True)
J = jacrev(f)(x)
print('Jacobian shape:', J.shape)

Jacobian shape: torch.Size([4, 4, 8])


## 4. Next step (not yet -- do this once the above all works)

LIBERO install is intentionally separate from this notebook to avoid dependency
conflicts with the newer PyTorch SmolVLA needs. Once cells 1-3 above run cleanly,
come back for the LIBERO install script.